In [1]:
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [2]:
movie_reviews.fileids()[:10] # 문서 아이디
movie_reviews.categories()
print( len( movie_reviews.fileids(categories='neg') ) )
print( len( movie_reviews.fileids(categories='pos') ) )

1000
1000


In [3]:
fileid = movie_reviews.fileids()[0]
movie_reviews.raw(fileid)[:100]
print(movie_reviews.categories(fileid))

['neg']


In [4]:
ids = movie_reviews.fileids()
reviews = [ movie_reviews.raw(id) for id in ids ]
categories = [ movie_reviews.categories(id)[0] for id in ids ]

In [ ]:
# TexxtBlob을 이용한 감성 분석

# 단어별 감성점수의 평균을 구하고, 강조어 (very, extremely)나 부정어(not)에 대한 처리 규칙이 포함되어 있음

In [11]:
from textblob import TextBlob
result = TextBlob(reviews[0])
result.sentiment

# polarity -1 ~ 1 : 0->중립, 1->매우 긍정
# subbjectivity 0 ~ 1 : 1 -> 매우 주관적, 0 -> 객관적

Sentiment(polarity=0.06479782948532947, subjectivity=0.5188408350908352)

In [15]:
from sklearn.metrics import accuracy_score
predict = [ TextBlob(review).sentiment for review in reviews ]

In [17]:
import numpy as np
textblob_predict = [ pred[0] for pred in predict ]
textblob_predict = np.array(textblob_predict)> 0.1
textblob_predict = [ 'pos' if pred == True else 'neg' for pred in textblob_predict  ]
accuracy_score(categories, textblob_predict)

0.746

In [ ]:
# 특징 : 주관성 지수를 함께 제공
# 리뷰 중에서 주관적과 객관적의 비율 (혼자 해보기)

In [19]:
# AFINN을 이용한 감성 분석
# 감성점수를 -5 ~ +5 점수 체계 : 처리속도가 빠름
from afinn import Afinn

In [22]:
afn = Afinn(emoticons=True)
result = np.array([afn.score(review) for review in reviews]) > 0
result = ['pos' if re == True else 'neg' for re in result]
accuracy_score(categories, result)

0.664

In [29]:
# VADER를 이용한 감성 분석
# 대문자, 구두점, 이모티콘, sns 데이터 특유의 감성표현을 포착하는 '문법적 휴리스틱' 원리를 기술
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
analyzer = SentimentIntensityAnalyzer()
for sentence in tqdm(reviews):
    vs = analyzer.polarity_scores(sentence)
    # print("{:-<65} {}".format(sentence[:10], str(vs)))
    if vs['compound'] > 0:
        result.append('pos')
    else:
        result.append('neg')

100%|██████████| 2000/2000 [00:25<00:00, 79.29it/s] 


In [39]:
import re
from konlpy.tag import Okt
class CustomTextBlob:
    """TextBlob 모사
        1. 사전기반 점수 합산
        2. 부정어 처리
        3. 긍정어 처리
        4. 평균점수 산출
    """
    def __init__(self, language='en'):
        self.language = language
        self.okt = Okt() if language == 'ko'else None
        # 감정어 사전(원래는 수만개)
        self.lexicon = {
            'great' : 0.8, 'good' : 0.5, 'happy' : 0.7,
            'bad' : -0.6, 'terrible' : -0.9, 'sad' : -0.5,
            '좋다' : 0.6, '최고' : 0.9, '행복하다' : 0.8,
            '나쁘다' : -0.6, '최악' : -0.9, '슬프다' : -0.7
        }
        # 부정어, 강조거
        self.negation = {
            'not', 'never', 'no', '안', '못', '아니'
        }
        self.intensifiers = {
            'very':1.5, '매우' : 1.5
        }
    
    def tokenize(self, text):
        if self.language == 'ko':
            # 한국어는 형태소 분석 후 원형(stem) 추출
            return [ word for word, pos in self.okt.pos(text,stem=True) ]
        else: 
            # 단순 공백 및 특수문자 제거
            return re.findall(r'\w+', text.lower()) # [a-zA-Z0-9]
    def analyze(self, text):
        tokens = self.tokenize(text)
        scores = []
        negate = False
        intensity = 1.0
        for i, token in enumerate(tokens):
            # 강조어 확인
            if token in self.intensifiers:
                intensity = self.intensifiers[token]
                continue
            # 부정어 확인
            if token in self.negation:
                negate = True
                continue
            # 감정단어 점수 계산
            if token in self.lexicon:
                score = self.lexicon[token] * intensity
                # 부정어 처리
                if negate:
                    score *= -0.5
                    negate = False # 적용후 해제
                scores.append(score)
                intensity = 1.0 # 적용후 초기화
        if not scores:
            return 0.0
        # 평균 산출
        return sum(scores) / len(scores)

In [40]:
en_blob = CustomTextBlob()
en_blob.analyze('this movie is good')

0.5